# CardioIA – Fase 2 | Parte 1: extração de sintomas e sugestão de diagnóstico

Este notebook lê os relatos de pacientes em `frases_sintomas.txt`, procura neles as expressões do mapa de conhecimento (`mapa_conhecimento.csv`) e sugere um diagnóstico.

O processo tem três etapas:

1. **Normalização:** frases e sintomas vão para minúsculas e perdem os acentos, para que "tórax" e "torax" sejam a mesma palavra.
2. **Extração:** cada expressão do mapa (colunas `sintoma_1` e `sintoma_2`) é procurada dentro da frase.
3. **Pontuação:** cada doença ganha 1 ponto por sintoma diferente encontrado. A doença com mais pontos vira o diagnóstico sugerido, e as outras aparecem como hipóteses secundárias.

As doenças do mapa foram escolhidas para conversar com a Fase 1: infarto (classe **MI** do PTB-XL), hipertrofia (**HYP**, aqui como cardiomiopatia hipertrófica), distúrbios de ritmo e condução (**CD**, aqui fibrilação atrial e bradicardia) e as anginas da planilha `cardioia_dados_numericos_iot.xlsx`.

> Projeto acadêmico. As sugestões não substituem avaliação médica.

In [1]:
import unicodedata
import pandas as pd

pd.set_option("display.max_colwidth", None)

## 1. Leitura dos arquivos

In [2]:
with open("frases_sintomas.txt", encoding="utf-8") as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

mapa = pd.read_csv("mapa_conhecimento.csv")

print(f"{len(frases)} frases carregadas")
print(f"{len(mapa)} associações para {mapa['doenca_associada'].nunique()} doenças")
mapa.head(10)

10 frases carregadas
34 associações para 10 doenças


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no tórax,Infarto agudo do miocárdio
1,dor forte no peito,dor no peito em repouso,Infarto agudo do miocárdio
2,irradia para o braço,dor no braço esquerdo,Infarto agudo do miocárdio
3,suor frio,náusea,Infarto agudo do miocárdio
4,dor na mandíbula,sensação de morte,Infarto agudo do miocárdio
5,falta de ar,dificuldade para respirar,Angina estável
6,aperto no peito,dor no peito ao esforço,Angina estável
7,subo escadas,piora com esforço,Angina estável
8,melhora com repouso,passa quando descanso,Angina estável
9,cansaço constante,fadiga,Insuficiência cardíaca


## 2. Ontologia: de sintomas para doenças

Cada linha do CSV liga duas expressões a uma doença. Aqui as duas colunas viram uma lista única de pares **(sintoma → doença)**, que é o que o extrator consulta. Uma mesma expressão pode apontar para mais de uma doença (por exemplo, "palpitações" aparece em fibrilação atrial, miocardite e cardiomiopatia hipertrófica). Quem desempata são os outros sintomas da frase.

In [3]:
ontologia = (
    mapa.melt(id_vars="doenca_associada", value_name="sintoma")[["sintoma", "doenca_associada"]]
    .drop_duplicates()
)

ontologia.groupby("doenca_associada")["sintoma"].apply(", ".join).to_frame("sintomas")

,sintomas
doenca_associada,
Angina estável,"falta de ar, aperto no peito, subo escadas, melhora com repouso, dificuldade para respirar, dor no peito ao esforço, piora com esforço, passa quando descanso"
Bradicardia,"batimentos lentos, tontura, quase caí, pulso fraco, fraqueza, vista escura"
Cardiomiopatia hipertrófica,"desmaiei, morte súbita, durante o treino, desmaio durante exercício, falta de ar ao correr, palpitações"
Embolia pulmonar,"falta de ar súbita, panturrilha inchada, viagem de avião, dor ao respirar, perna inchada, tosse com sangue"
Fibrilação atrial,"coração disparado, batimentos irregulares, coração falhando, palpitações, coração acelerado, pulso irregular"
Hipertensão arterial,"dor de cabeça na nuca, visão embaçada, sangramento no nariz, pressão alta, zumbido no ouvido, dor de cabeça forte"
Infarto agudo do miocárdio,"dor no peito, dor forte no peito, irradia para o braço, suor frio, dor na mandíbula, aperto no tórax, dor no peito em repouso, dor no braço esquerdo, náusea, sensação de morte"
Insuficiência cardíaca,"cansaço constante, pernas inchadas, falta de ar ao deitar, vários travesseiros, fadiga, tornozelos inchados, acordo sem ar, falta de ar aos esforços"
Miocardite,"virose, cansaço extremo, depois de uma gripe, febre, palpitações, dor no peito"


## 3. Extração de sintomas e sugestão de diagnóstico

In [4]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def diagnosticar(frase):
    texto = normalizar(frase)
    encontrados = ontologia[ontologia["sintoma"].map(lambda sintoma: normalizar(sintoma) in texto)]
    pontos = encontrados["doenca_associada"].value_counts()
    return {
        "frase": frase,
        "sintomas_identificados": list(encontrados["sintoma"].unique()),
        "diagnostico_sugerido": pontos.index[0] if len(pontos) else "Sem correspondência no mapa",
        "pontos": int(pontos.iloc[0]) if len(pontos) else 0,
        "outras_hipoteses": [f"{doenca} ({qtd})" for doenca, qtd in pontos.iloc[1:].items()],
    }


resultados = [diagnosticar(frase) for frase in frases]

for numero, r in enumerate(resultados, 1):
    print(f"Paciente {numero}: {r['frase']}")
    print(f"  Sintomas identificados: {', '.join(r['sintomas_identificados'])}")
    print(f"  Diagnóstico sugerido:   {r['diagnostico_sugerido']} ({r['pontos']} sintomas)")
    if r["outras_hipoteses"]:
        print(f"  Outras hipóteses:       {', '.join(r['outras_hipoteses'])}")
    print()

Paciente 1: Há duas horas sinto uma dor forte no peito que irradia para o braço esquerdo, com suor frio e náusea, e precisei parar de trabalhar.
  Sintomas identificados: dor forte no peito, irradia para o braço, suor frio, náusea
  Diagnóstico sugerido:   Infarto agudo do miocárdio (4 sintomas)

Paciente 2: Há três semanas sinto um aperto no peito sempre que subo escadas, que melhora com repouso, e agora evito ir a pé até o mercado.
  Sintomas identificados: aperto no peito, subo escadas, melhora com repouso
  Diagnóstico sugerido:   Angina estável (3 sintomas)

Paciente 3: Há um mês estou com as pernas inchadas no fim do dia, sinto falta de ar ao deitar e preciso dormir com vários travesseiros para conseguir respirar.
  Sintomas identificados: falta de ar, pernas inchadas, falta de ar ao deitar, vários travesseiros
  Diagnóstico sugerido:   Insuficiência cardíaca (3 sintomas)
  Outras hipóteses:       Angina estável (1)

Paciente 4: Desde a semana passada sinto o coração disparado e 

In [5]:
tabela = pd.DataFrame(resultados)
tabela.index = range(1, len(tabela) + 1)
tabela[["diagnostico_sugerido", "pontos", "outras_hipoteses"]]

,diagnostico_sugerido,pontos,outras_hipoteses
1,Infarto agudo do miocárdio,4,[]
2,Angina estável,3,[]
3,Insuficiência cardíaca,3,[Angina estável (1)]
4,Fibrilação atrial,2,[]
5,Bradicardia,4,[]
6,Hipertensão arterial,3,[]
7,Pericardite,3,"[Miocardite (2), Infarto agudo do miocárdio (1)]"
8,Miocardite,4,"[Fibrilação atrial (1), Pericardite (1), Cardiomiopatia hipertrófica (1)]"
9,Cardiomiopatia hipertrófica,4,[Angina estável (1)]
10,Embolia pulmonar,3,[Angina estável (1)]


## 4. Validação

Cada frase foi escrita pensando em uma doença. O gabarito abaixo confere se o sistema chegou nela.

In [6]:
gabarito = [
    "Infarto agudo do miocárdio",
    "Angina estável",
    "Insuficiência cardíaca",
    "Fibrilação atrial",
    "Bradicardia",
    "Hipertensão arterial",
    "Pericardite",
    "Miocardite",
    "Cardiomiopatia hipertrófica",
    "Embolia pulmonar",
]

tabela["esperado"] = gabarito
tabela["acertou"] = tabela["diagnostico_sugerido"] == tabela["esperado"]
print(f"Acertos: {tabela['acertou'].sum()}/{len(tabela)}")
tabela[["esperado", "diagnostico_sugerido", "acertou"]]

Acertos: 10/10


,esperado,diagnostico_sugerido,acertou
1,Infarto agudo do miocárdio,Infarto agudo do miocárdio,True
2,Angina estável,Angina estável,True
3,Insuficiência cardíaca,Insuficiência cardíaca,True
4,Fibrilação atrial,Fibrilação atrial,True
5,Bradicardia,Bradicardia,True
6,Hipertensão arterial,Hipertensão arterial,True
7,Pericardite,Pericardite,True
8,Miocardite,Miocardite,True
9,Cardiomiopatia hipertrófica,Cardiomiopatia hipertrófica,True
10,Embolia pulmonar,Embolia pulmonar,True


## 5. Limitações: testando frases fora do padrão

A busca por expressões é simples e transparente, mas tem pontos cegos. Os três testes abaixo mostram isso:

In [7]:
testes = [
    "Não sinto dor no peito nem falta de ar, vim só fazer um check-up.",
    "Estou com precordialgia e dispneia desde ontem.",
    "Sinto uma dor no tórax que aperta e vai até o braço.",
]

for frase in testes:
    r = diagnosticar(frase)
    print(frase)
    print(f"  -> {r['diagnostico_sugerido']} | sintomas: {r['sintomas_identificados']}")
    print()

Não sinto dor no peito nem falta de ar, vim só fazer um check-up.
  -> Infarto agudo do miocárdio | sintomas: ['dor no peito', 'falta de ar']

Estou com precordialgia e dispneia desde ontem.
  -> Sem correspondência no mapa | sintomas: []

Sinto uma dor no tórax que aperta e vai até o braço.
  -> Sem correspondência no mapa | sintomas: []



O que os testes mostram:

- **Negação:** "não sinto dor no peito" contém "dor no peito", então o sistema sugere infarto para alguém saudável. A busca por texto não entende o "não".
- **Vocabulário técnico:** "precordialgia" e "dispneia" querem dizer dor no peito e falta de ar, mas não estão no mapa. Resultado: nenhuma correspondência.
- **Variação de escrita:** "dor no tórax que aperta" descreve a mesma coisa que "aperto no tórax", mas as palavras estão em outra ordem e a busca exata falha.

Um mapa com mais sinônimos resolve parte disso. O resto pede técnicas de NLP mais robustas, como detecção de negação, lematização ou embeddings. Por isso um sistema assim só serve para **apoiar** a triagem, sempre com revisão de um profissional de saúde.